In [0]:
pip install langchain langchain-community==0.0.28 langchain-huggingface chromadb==0.4.22 sentence-transformers pypdf openai langchain-openai python-dotenv rank_bm25

In [0]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever

# 1. Load PDF
pdf_files = ["/Workspace/Users/surbhiwahie@gmail.com/Rag-System---Ask-My-Doc/Data/raw/Diabetes Care Booklet.pdf"]
all_docs = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    
    # Add source info
    for d in docs:
        d.metadata["source"] = file.split("/")[-1]
    
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages total")

In [0]:
# 2. Chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(all_docs)

print(f"Created {len(chunks)} chunks")

In [0]:
import os
import shutil
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY"] = "False"

# Delete old DB if exists
if os.path.exists("/tmp/chroma_db"):
    shutil.rmtree("/tmp/chroma_db")


# Add metadata
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i

# Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Store in Chroma
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)


# Retriever
vector_retriever = db.as_retriever(search_kwargs={"k": 5})

print("✅ Vector DB created!")

In [0]:
# IMPORTANT: you need access to chunks
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

In [0]:
def hybrid_retrieve(query, vector_retriever, bm25_retriever, k=5):
    
    # 1. Get results from both
    vector_docs = vector_retriever.invoke(query)
    bm25_docs = bm25_retriever.invoke(query)
    
    # 2. Combine
    all_docs = vector_docs + bm25_docs
    
    # 3. Remove duplicates (IMPORTANT)
    unique_docs = {}
    
    for doc in all_docs:
        key = doc.page_content  # unique text
        if key not in unique_docs:
            unique_docs[key] = doc
    
    # 4. Return top K
    return list(unique_docs.values())[:k]

In [0]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [0]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",   # cheaper + fast (good for your project)
    temperature=0,         # important → reduces hallucination
    api_key=api_key
)

In [0]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=3):
    
    pairs = [(query, doc.page_content) for doc in docs]
    
    scores = reranker.predict(pairs)
    
    # Combine docs with scores
    scored_docs = list(zip(docs, scores))
    
    # Sort by score (high → low)
    ranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    
    # Return top_k docs
    return [doc for doc, score in ranked_docs[:top_k]]

In [0]:
def answer_question(query):
    
    # 1. Retrieve relevant chunks
    docs = hybrid_retrieve(query, vector_retriever, bm25_retriever)

    docs = rerank(query, docs, top_k=3)
    
    # 2. Build context
    context = "\n\n".join([
    f"[Source: {d.metadata.get('source')}, Page: {d.metadata.get('page')}]\n{d.page_content}"
    for d in docs])


    
    # 3. Strict prompt (VERY IMPORTANT)
    prompt = f"""
You are a strict AI assistant.

ONLY answer using the provided context.
If the answer is not in the context, say:
"I cannot find sufficient information in the provided documents."

Always include citations in this format:
[Source: filename, Page X]

Context:
{context}

Question:
{query}

Answer:
"""
    
    # 4. Call LLM
    response = llm.invoke(prompt)
    
    return response.content

In [0]:
query = "what are symptoms of diabetes?"

answer = answer_question(query)

print(answer)